In [ ]:
!pip install haversine

In [ ]:
import pandas as pd
import numpy as np
import os
from haversine import haversine, Unit

# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# Define paths
base_path = '/content/drive/MyDrive/FraudDetection_project/datasets/'
processed_path = '/content/drive/MyDrive/FraudDetection_project/processed_data/'

# Ensure processed directory exists
os.makedirs(processed_path, exist_ok=True)

print("Loading raw datasets... This might take a moment.")
sparkov_train = pd.read_csv(base_path + 'fraudTrain.csv')
sparkov_test = pd.read_csv(base_path + 'fraudTest.csv')
paysim_data = pd.read_csv(base_path + 'Paysim.csv')

# Combine Sparkov temporarily ONLY for the continuous rolling window calculation (Boundary Drop fix)
sparkov_train['is_test_set'] = 0
sparkov_test['is_test_set'] = 1
sparkov_full = pd.concat([sparkov_train, sparkov_test], ignore_index=True)

print("Data loaded successfully.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading raw datasets... This might take a moment.
Data loaded successfully.


In [ ]:
print("Starting Sparkov Feature Engineering...")

# 1. Parse Dates
sparkov_full['trans_date_trans_time'] = pd.to_datetime(sparkov_full['trans_date_trans_time'])
sparkov_full['dob'] = pd.to_datetime(sparkov_full['dob'])

# Sort strictly by time to ensure rolling windows are accurate
sparkov_full = sparkov_full.sort_values(['cc_num', 'trans_date_trans_time']).reset_index(drop=True)

# 2. transaction_hour
sparkov_full['transaction_hour'] = sparkov_full['trans_date_trans_time'].dt.hour

# 3. customer_age
sparkov_full['customer_age'] = (sparkov_full['trans_date_trans_time'] - sparkov_full['dob']).dt.days // 365

# 4. distance_km (Fast Vectorized Haversine)
def vectorized_haversine(lat1, lon1, lat2, lon2):
    R = 6371.0 # Earth radius in kilometers
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

sparkov_full['distance_km'] = vectorized_haversine(
    sparkov_full['lat'], sparkov_full['long'],
    sparkov_full['merch_lat'], sparkov_full['merch_long']
)

# 5. txn_velocity (Rolling 24-hour count per customer)
# Because we concatenated the data and sorted by time, there is no "boundary drop" at the start of 2020.
sparkov_full = sparkov_full.set_index('trans_date_trans_time')
sparkov_full['txn_velocity'] = sparkov_full.groupby('cc_num')['amt'].transform(lambda x: x.rolling('24h').count())
sparkov_full = sparkov_full.reset_index()

# --- ENFORCING STRICT TRAINING DATA LEAKAGE PREVENTION ---
train_mask = sparkov_full['is_test_set'] == 0

# 6. customer_avg_amount & amount_deviation
# Learn averages ONLY from 2019 data
customer_avg_dict = sparkov_full[train_mask].groupby('cc_num')['amt'].mean()
global_median_amt = sparkov_full[train_mask]['amt'].median() # Cold start fallback

# Map to full dataset, fill unknowns with median
sparkov_full['customer_avg_amount'] = sparkov_full['cc_num'].map(customer_avg_dict).fillna(global_median_amt)
sparkov_full['amount_deviation'] = (sparkov_full['amt'] - sparkov_full['customer_avg_amount']) / (sparkov_full['customer_avg_amount'] + 1)

# 7. category_fraud_rate
# Learn fraud rates ONLY from 2019 data
category_fraud_dict = sparkov_full[train_mask].groupby('category')['is_fraud'].mean()
global_fraud_mean = sparkov_full[train_mask]['is_fraud'].mean() # Cold start fallback

# Map to full dataset, fill unknowns with global mean
sparkov_full['category_fraud_rate'] = sparkov_full['category'].map(category_fraud_dict).fillna(global_fraud_mean)

print("Sparkov Feature Engineering Complete.")

Starting Sparkov Feature Engineering...
Sparkov Feature Engineering Complete.


In [ ]:
print("Starting PaySim Feature Engineering...")

# --- SELF-HEALING CHECK ---
# If step got trapped in the index from a previous failed run, this pulls it back out
if paysim_data.index.name == 'step':
    print("Recovering 'step' from index...")
    paysim_data = paysim_data.reset_index()

# If step is still missing, we print the columns to see what went wrong
if 'step' not in paysim_data.columns:
    print("\nCRITICAL: 'step' is completely missing! Here are the columns Colab sees:")
    print(paysim_data.columns.tolist())
    raise KeyError("The 'step' column is missing from the dataset. Please restart the runtime and run all cells from the top.")

# Sort by step (time) to ensure rolling windows are accurate
paysim_data = paysim_data.sort_values(['nameOrig', 'step']).reset_index(drop=True)

# 1. balance_drained
paysim_data['balance_drained'] = (paysim_data['newbalanceOrig'] == 0).astype(int)

# 2. balance_drop_ratio
paysim_data['balance_drop_ratio'] = (paysim_data['oldbalanceOrg'] - paysim_data['newbalanceOrig']) / (paysim_data['oldbalanceOrg'] + 1)

# 3. is_transfer_or_cashout
paysim_data['is_transfer_or_cashout'] = paysim_data['type'].isin(['TRANSFER', 'CASH_OUT']).astype(int)

# 4. receiver_balance_unchanged
# Apply 1 ONLY if it's a TRANSFER and the balance didn't move. Otherwise, 0.
paysim_data['receiver_balance_unchanged'] = (
    (paysim_data['newbalanceDest'] == paysim_data['oldbalanceDest']) &
    (paysim_data['type'] == 'TRANSFER')
).astype(int)

# 5. txn_velocity (Optimized for speed)
# We use cumcount() which is fully vectorized. Because the data is already
# sorted by nameOrig and step, this safely counts how many times this
# sender has transacted up to this point, running in ~1 second.
paysim_data['txn_velocity'] = paysim_data.groupby('nameOrig').cumcount() + 1

print("PaySim Feature Engineering Complete.")

Starting PaySim Feature Engineering...
Recovering 'step' from index...
PaySim Feature Engineering Complete.


In [ ]:
print("Saving engineered datasets to Drive...")

# Split Sparkov back into Train (2019) and Test (2020)
sparkov_train_engineered = sparkov_full[sparkov_full['is_test_set'] == 0].drop(columns=['is_test_set'])
sparkov_test_engineered = sparkov_full[sparkov_full['is_test_set'] == 1].drop(columns=['is_test_set'])

# Save Sparkov
sparkov_train_engineered.to_csv(processed_path + 'sparkov_train_engineered.csv', index=False)
sparkov_test_engineered.to_csv(processed_path + 'sparkov_test_engineered.csv', index=False)

# Save PaySim (We haven't split this yet, Day 4 will handle the step-aware split)
paysim_data.to_csv(processed_path + 'paysim_engineered.csv', index=False)

print("Day 3 Deliverables successfully saved to /processed_data/")
print(f"Sparkov Train Shape: {sparkov_train_engineered.shape}")
print(f"Sparkov Test Shape: {sparkov_test_engineered.shape}")
print(f"PaySim Shape: {paysim_data.shape}")

Saving engineered datasets to Drive...
Day 3 Deliverables successfully saved to /processed_data/
Sparkov Train Shape: (1296675, 30)
Sparkov Test Shape: (555719, 30)
PaySim Shape: (6362620, 16)
